# Parallelization Workflows

**The problem:** cramming a complex decision into one prompt makes Claude juggle many competing criteria at once → less reliable. Example: a *material designer* that recommends the best material for a part (metal / polymer / ceramic / composite / elastomer / wood). One giant prompt with all the criteria overloads the model.

**The pattern — parallelization:**
1. **Split** the task into focused sub-tasks (one per material, each with specialized criteria)
2. **Run** them in parallel (independent calls)
3. **Aggregate** the results into a final decision

Sub-tasks needn't be identical — each can have its own prompt/tools/criteria.

> **Adaptation:** the course sends an *image* of a part. To keep this runnable without an asset, we use a **text part-spec**; the workflow structure is identical. Runs on Haiku 4.5, fanning out with a thread pool.

## Setup

In [1]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

import concurrent.futures
from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5-20251001"


def get_text(message):
    for block in message.content:
        if block.type == "text":
            return block.text
    return ""


def chat(prompt, system=None, max_tokens=500):
    params = {"model": model, "max_tokens": max_tokens,
              "messages": [{"role": "user", "content": prompt}], "temperature": 1.0}
    if system:
        params["system"] = system
    return get_text(client.messages.create(**params))


print(f"Ready. Model: {model}")

Ready. Model: claude-haiku-4-5-20251001


## The task + specialized criteria per material

One part spec; a dict of materials, each with the criteria that matter *for that material*. This is what would otherwise be one overloaded mega-prompt.

In [2]:
PART_SPEC = """
A bracket for an engine bay: must withstand sustained 180C, bear a 4kN static load,
resist road-salt corrosion, tolerate vibration/fatigue over 10 years, and stay as light
as possible. Produced at ~50,000 units/year.
"""

MATERIALS = {
    "metal": "strength-to-weight, fatigue life, corrosion resistance (needs coating?), machinability, cost at 50k/yr",
    "polymer": "max continuous service temperature vs 180C, creep under sustained load, UV/chemical resistance, injection-molding fit",
    "ceramic": "brittleness / fracture toughness under vibration, thermal shock, load-bearing capacity, manufacturability of the geometry",
    "composite": "fiber/matrix temp limits, anisotropic strength, fatigue, tooling/cost at volume, joining to metal parts",
    "elastomer": "load-bearing suitability (usually poor for structural), temperature range, compression set, vibration damping",
    "wood": "structural/thermal viability in an engine bay at all (likely disqualifying), moisture/heat behavior",
}

## Step 1+2 — one focused evaluation per material, in parallel

Each call sees only *its* material's criteria, so Claude concentrates on a single assessment. We fan them out with a thread pool (independent, so they run concurrently).

In [3]:
def evaluate_material(name, criteria):
    prompt = f"""
You are a materials engineer. Assess ONLY whether {name.upper()} is a good choice for this part.
Use these {name}-specific criteria: {criteria}

Part requirements:
{PART_SPEC}

Respond with:
- Suitability score (1-10)
- 2 sentences of justification focused strictly on {name}.
"""
    return name, chat(prompt)


with concurrent.futures.ThreadPoolExecutor(max_workers=len(MATERIALS)) as ex:
    futures = [ex.submit(evaluate_material, n, c) for n, c in MATERIALS.items()]
    evaluations = [f.result() for f in futures]

for name, verdict in evaluations:
    print(f"===== {name.upper()} =====\n{verdict}\n")

===== METAL =====
**Suitability Score: 7/10**

**Justification:**

Aluminum alloys (6061-T6 or 7075-T6) or stainless steel (304/316) offer excellent strength-to-weight ratios and proven fatigue performance for engine-bay vibration environments, with aluminum particularly light and machinable at 50k/year production volumes. However, the 180°C sustained temperature requirement pushes near the limits of standard aluminum (which softens above 150°C) and demands either premium heat-resistant alloys or stainless steel, both raising material and processing costs; additionally, corrosion protection via anodizing (aluminum) or passivation (stainless) adds secondary operations that increase complexity at this production scale.

===== POLYMER =====
**Suitability Score: 3/10**

Polymer is unsuitable for this application. The sustained 180°C service requirement immediately disqualifies most engineering polymers—only specialized high-temperature variants (PEEK, PPS) operate reliably at this threshol

## Step 3 — aggregate into a final recommendation

Feed all the independent verdicts to one final call that compares them and picks a winner. The aggregator sees concise, specialized analyses rather than raw criteria — easier to reason over.

In [4]:
combined = "\n\n".join(f"## {name.upper()}\n{verdict}" for name, verdict in evaluations)

agg_prompt = f"""
Below are independent suitability evaluations of one part for different materials.
Compare them and recommend the SINGLE best material, with a short justification and the runner-up.

Part requirements:
{PART_SPEC}

Evaluations:
{combined}
"""

print(chat(agg_prompt, max_tokens=600))

# Recommendation

## **BEST CHOICE: METAL (Aluminum 7075-T6 or Stainless Steel 304)**

**Justification:**
Metal is the clear winner despite a 7/10 score. It is the **only material that reliably satisfies all hard constraints**: sustained 180°C operation (via premium aluminum alloys or stainless steel), proven 4kN static load capacity, inherent road-salt corrosion resistance (especially stainless), and a 10-year vibration fatigue track record in automotive engine bays. At 50,000 units/year, metal tooling and secondary finishing (anodizing/passivation) are economically justified and well-established. The cost and complexity penalties are acceptable trade-offs for safety and durability in a load-bearing, high-temperature bracket.

---

## **Runner-up: COMPOSITE (Carbon-Fiber Epoxy)**

**Justification:**
Composites score 6/10 and offer attractive weight savings and corrosion resistance, but the 180°C sustained temperature requirement forces reliance on expensive, less-mature high-temperatu

## Why this works

- **Focused attention** — each call weighs one material's criteria, not six at once → more thorough, more accurate.
- **Easier optimization** — tune the metal prompt without touching the others; test each independently.
- **Scalability** — add a material = add one parallel call; no rewriting existing prompts.
- **Reliability** — lower cognitive load per call → more consistent results.
- **Speed** — independent sub-tasks run concurrently, so wall-clock ≈ the slowest single call, not the sum.

**When to use it:** a complex decision that decomposes into *independent* evaluations — multiple criteria, several options to compare, or different domains of expertise. Each sub-task must stand alone and contribute a distinct piece.

Next: **Chaining workflows** — where each step's output *feeds* the next (the opposite of independent).